# 01 — Data Audit

Audit file `Data/Raw/Data_input.xlsx` theo workflow mới của project **Top Place for repair → chọn Top 3 khu ưu tiên mở Trạm iCare**.

Notebook này chỉ **kiểm tra dữ liệu**, không chọn Top 3 và không tạo score.

Guardrails:
- Missing ≠ 0.
- Competitor count = market-activity evidence, không phải demand thật.
- Traffic evidence count ≠ traffic volume.
- Apartment-price proxy ≠ income.
- Coverage distance tại verified competitor points ≠ district-wide catchment.
- Universe screening phải giữ đúng 19 `district_old` của project cũ.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

CANDIDATES = [
    Path("../Data/Raw/Data_input.xlsx"),
    Path("Data/Raw/Data_input.xlsx"),
    Path("Data_input.xlsx"),
    Path("/mnt/data/Data_input.xlsx"),
]

def inspect_candidate(path):
    if not path.exists():
        return None
    try:
        xls = pd.ExcelFile(path)
        result = {
            "path": path,
            "sheet_names": xls.sheet_names,
            "score": 0,
            "issues": [],
        }

        required_checks = {
            "04_Market_Locations": "district_old",
            "09_District_Opportunity_Base": "district_old",
            "14_Complete_Screening_Core": "district_old",
        }

        for sheet, required_col in required_checks.items():
            if sheet not in xls.sheet_names:
                result["issues"].append(f"missing sheet {sheet}")
                continue

            header = pd.read_excel(path, sheet_name=sheet, nrows=0)
            cols = [str(c).strip().replace("\ufeff", "") for c in header.columns]
            if required_col in cols:
                result["score"] += 1
            elif required_col == "district_old" and "district" in cols:
                # accepted alias; normalized after loading
                result["score"] += 0.5
                result["issues"].append(f"{sheet}: uses 'district' alias instead of 'district_old'")
            else:
                result["issues"].append(
                    f"{sheet}: missing required column {required_col}; columns={cols}"
                )
        return result
    except Exception as e:
        return {
            "path": path,
            "sheet_names": [],
            "score": -1,
            "issues": [f"cannot read workbook: {e}"],
        }

candidate_reports = [r for p in CANDIDATES if (r := inspect_candidate(p)) is not None]

assert candidate_reports, (
    "Không tìm thấy Data_input.xlsx. "
    "Hãy đặt file tại ../Data/Raw/Data_input.xlsx hoặc cùng thư mục notebook."
)

print("=== DATA_INPUT CANDIDATES ===")
for r in candidate_reports:
    print(f"{str(r['path']):40s} score={r['score']}")
    for issue in r["issues"]:
        print("   -", issue)

best_score = max(r["score"] for r in candidate_reports)
best = [r for r in candidate_reports if r["score"] == best_score]

# If several copies are equally valid, prefer the newest one.
best.sort(key=lambda r: r["path"].stat().st_mtime, reverse=True)
selected = best[0]
file_path = selected["path"]

if best_score < 3:
    raise ValueError(
        "Không có Data_input.xlsx nào đạt schema đầy đủ cho district audit.\n"
        "Bạn có thể đang chạy nhầm/stale workbook.\n"
        f"Best candidate: {file_path.resolve()}\n"
        f"Issues: {selected['issues']}"
    )

print("\nSELECTED INPUT:", file_path.resolve())
print("Schema score:", best_score, "/ 3")


=== DATA_INPUT CANDIDATES ===
..\Data\Raw\Data_input.xlsx              score=3

SELECTED INPUT: D:\IcareWork\Place for icare\Data\Raw\Data_input.xlsx
Schema score: 3 / 3


In [2]:
xls = pd.ExcelFile(file_path)

# Excel giới hạn tên sheet tối đa 31 ký tự.
# Một số sheet dài có thể bị cắt tên khi lưu, nên resolve theo prefix.
sheet_prefixes = {
    "01_Current_Admin": "01_Current_Admin",
    "02_Evidence_Log": "02_Evidence_Log",
    "03_District_Profile": "03_District_Profile",
    "04_Market_Locations": "04_Market_Locations",
    "05_District_Market_Footprint": "05_District_Market_Footprint",
    "06_iCare_Coverage_Evidence": "06_iCare_Coverage_Evidence",
    "07_District_Traffic_Evidence": "07_District_Traffic_Evidence",
    "08_District_Traffic_Summary": "08_District_Traffic_Summary",
    "09_District_Opportunity_Base": "09_District_Opportunity_Base",
    "10_Gaps_Next": "10_Gaps_Next",
    "11_Old_Project_Universe_Research": "11_Old_Project_Universe_Resear",
    "12_Old_Project_Universe_Method": "12_Old_Project_Universe_Metho",
    "13_Additional_Context_Evidence": "13_Additional_Context_Evidenc",
    "14_Complete_Screening_Core": "14_Complete_Screening_Core",
}

def resolve_sheet_name(canonical_name, prefix):
    if canonical_name in xls.sheet_names:
        return canonical_name
    matches = [s for s in xls.sheet_names if s.startswith(prefix)]
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise ValueError(f"Ambiguous sheet prefix {prefix!r}: {matches}")
    raise KeyError(f"Sheet not found for {canonical_name!r}; expected prefix {prefix!r}")

resolved_sheets = {
    canonical: resolve_sheet_name(canonical, prefix)
    for canonical, prefix in sheet_prefixes.items()
}

print("=== SHEETS FOUND ===")
for s in xls.sheet_names:
    print("-", s)

print("\n=== RESOLVED SHEET NAMES ===")
for canonical, actual in resolved_sheets.items():
    mark = "" if canonical == actual else "  <- Excel-truncated name"
    print(f"{canonical:35s} -> {actual}{mark}")

required_sheets = list(sheet_prefixes.keys())
missing_sheets = []

print("\nMissing required sheets:", missing_sheets)
print("Extra sheets:", [s for s in xls.sheet_names if s not in resolved_sheets.values()])

# Corridor-level sheets added after district screening.
# These are audited when present, but they are not required for the original 19-area screening pipeline.
optional_corridor_prefixes = {
    "15_Top3_Corridor_Research": "15_Top3_Corridor_Research",
    "19_Corridor_Rent_Evidence": "19_Corridor_Rent_Evidence",
    "22_Corridor_Rent_Summary": "22_Corridor_Rent_Summary",
    "23_Corridor_Research_Status": "23_Corridor_Research_Status",
    "24_Corridor_Research_Master": "24_Corridor_Research_Master",
    "25_Method_Quality": "25_Method_Quality",
    "26_Corridor_Consolidated": "26_Corridor_Consolidated",
    "36_Evidence_Sufficiency": "36_Evidence_Sufficiency",
}

resolved_optional_corridor = {}
for canonical, prefix in optional_corridor_prefixes.items():
    if canonical in xls.sheet_names:
        resolved_optional_corridor[canonical] = canonical
    else:
        matches = [s for s in xls.sheet_names if s.startswith(prefix)]
        resolved_optional_corridor[canonical] = matches[0] if len(matches) == 1 else None


=== SHEETS FOUND ===
- 01_Current_Admin
- 02_Evidence_Log
- 03_District_Profile
- 04_Market_Locations
- 05_District_Market_Footprint
- 06_iCare_Coverage_Evidence
- 07_District_Traffic_Evidence
- 08_District_Traffic_Summary
- 11_Old_Project_Universe_Researc
- 12_Old_Project_Universe_Method
- 13_Additional_Context_Evidence
- 09_District_Opportunity_Base
- 10_Gaps_Next
- 14_Complete_Screening_Core
- 15_Top3_Corridor_Research
- 16_Top3_Corridor_Evidence
- 17_Top3_Corridor_Method
- 18_Corridor_Context_Anchors
- 19_Corridor_Rent_Evidence
- 20_Corridor_Property_Proxy
- 21_Corridor_DeepDive_Readiness
- 22_Corridor_Rent_Summary
- 23_Corridor_Research_Status
- 24_Top5_Research_Audit
- 25_Top5_Data_Completeness
- GPS_Change_Log_20261020

=== RESOLVED SHEET NAMES ===
01_Current_Admin                    -> 01_Current_Admin
02_Evidence_Log                     -> 02_Evidence_Log
03_District_Profile                 -> 03_District_Profile
04_Market_Locations                 -> 04_Market_Locations
05_Di

In [3]:
def normalize_columns(df):
    df = df.copy()
    df.columns = [
        str(c).strip().replace("\ufeff", "")
        for c in df.columns
    ]
    return df

data = {
    canonical: normalize_columns(pd.read_excel(file_path, sheet_name=actual))
    for canonical, actual in resolved_sheets.items()
}

# Normalize legacy/current alias only where district_old is the intended district-level key.
district_old_sheets = {
    "04_Market_Locations",
    "05_District_Market_Footprint",
    "06_iCare_Coverage_Evidence",
    "08_District_Traffic_Summary",
    "09_District_Opportunity_Base",
    "11_Old_Project_Universe_Research",
    "14_Complete_Screening_Core",
}
for name in district_old_sheets:
    df = data[name]
    if "district_old" not in df.columns and "district" in df.columns:
        data[name] = df.rename(columns={"district": "district_old"})

print("=== SHAPE / KEY-COLUMN AUDIT ===")
for name, df in data.items():
    key_status = ""
    if name in district_old_sheets:
        key_status = f" | district_old={'YES' if 'district_old' in df.columns else 'NO'}"
    print(f"{name:35s} rows={len(df):3d} cols={len(df.columns):2d}{key_status}")

# Fail early with a useful message instead of a later pandas KeyError.
critical_schema = {
    "04_Market_Locations": ["district_old"],
    "09_District_Opportunity_Base": ["district_old"],
    "14_Complete_Screening_Core": ["district_old"],
}
schema_errors = []
for name, required_cols in critical_schema.items():
    missing = [c for c in required_cols if c not in data[name].columns]
    if missing:
        schema_errors.append(
            f"{name}: missing {missing}; actual columns={list(data[name].columns)}"
        )

if schema_errors:
    raise ValueError(
        "District audit schema lỗi. Có khả năng notebook đang đọc nhầm/stale Data_input.xlsx:\n"
        + "\n".join(schema_errors)
        + f"\n\nWorkbook đang đọc: {file_path.resolve()}"
    )


=== SHAPE / KEY-COLUMN AUDIT ===
01_Current_Admin                    rows= 20 cols=10
02_Evidence_Log                     rows=  4 cols=10
03_District_Profile                 rows=  5 cols=19
04_Market_Locations                 rows= 65 cols=14 | district_old=YES
05_District_Market_Footprint        rows= 19 cols=15 | district_old=YES
06_iCare_Coverage_Evidence          rows= 62 cols=13 | district_old=YES
07_District_Traffic_Evidence        rows= 36 cols= 9
08_District_Traffic_Summary         rows= 19 cols=10 | district_old=NO
09_District_Opportunity_Base        rows= 19 cols=28 | district_old=YES
10_Gaps_Next                        rows=  6 cols= 6
11_Old_Project_Universe_Research    rows= 19 cols=27 | district_old=YES
12_Old_Project_Universe_Method      rows=  8 cols= 2
13_Additional_Context_Evidence      rows= 23 cols=10
14_Complete_Screening_Core          rows= 19 cols=18 | district_old=YES


In [4]:
print("=== DUPLICATE ROW AUDIT ===")
for name, df in data.items():
    print(f"{name:35s} duplicate_rows={int(df.duplicated().sum())}")


=== DUPLICATE ROW AUDIT ===
01_Current_Admin                    duplicate_rows=0
02_Evidence_Log                     duplicate_rows=0
03_District_Profile                 duplicate_rows=0
04_Market_Locations                 duplicate_rows=0
05_District_Market_Footprint        duplicate_rows=0
06_iCare_Coverage_Evidence          duplicate_rows=0
07_District_Traffic_Evidence        duplicate_rows=0
08_District_Traffic_Summary         duplicate_rows=0
09_District_Opportunity_Base        duplicate_rows=0
10_Gaps_Next                        duplicate_rows=0
11_Old_Project_Universe_Research    duplicate_rows=0
12_Old_Project_Universe_Method      duplicate_rows=0
13_Additional_Context_Evidence      duplicate_rows=0
14_Complete_Screening_Core          duplicate_rows=0


In [5]:
admin = data["01_Current_Admin"]
evidence = data["02_Evidence_Log"]
locations = data["04_Market_Locations"]
footprint = data["05_District_Market_Footprint"]
coverage = data["06_iCare_Coverage_Evidence"]
traffic = data["07_District_Traffic_Evidence"]
traffic_summary = data["08_District_Traffic_Summary"]
opportunity = data["09_District_Opportunity_Base"]
research = data["11_Old_Project_Universe_Research"]
context = data["13_Additional_Context_Evidence"]
core = data["14_Complete_Screening_Core"]

print("Loaded audit tables.")


Loaded audit tables.


## Source traceability audit

Không giả định mọi sheet có cùng tên cột nguồn. Chỉ audit các source-column thực sự tồn tại.


In [6]:
source_checks = [
    ("01_Current_Admin", admin, ["Nguồn chính"]),
    ("02_Evidence_Log", evidence, ["Nguồn", "source_url", "Source URL"]),
    ("04_Market_Locations", locations, ["source_url"]),
    ("07_District_Traffic_Evidence", traffic, ["Source URL"]),
    ("11_Old_Project_Universe_Research", research, [
        "population_source", "enterprise_source", "income_source", "pp_source"
    ]),
    ("13_Additional_Context_Evidence", context, ["source_url"]),
]

print("=== SOURCE AUDIT ===")
for sheet_name, df, candidates in source_checks:
    existing = [c for c in candidates if c in df.columns]
    if not existing:
        print(f"{sheet_name:35s} no recognized source column -> manual/structure check only")
        continue
    for col in existing:
        missing = int(df[col].isna().sum())
        print(f"{sheet_name:35s} {col:24s} missing={missing:3d} / {len(df)}")


=== SOURCE AUDIT ===
01_Current_Admin                    Nguồn chính              missing=  0 / 20
02_Evidence_Log                     no recognized source column -> manual/structure check only
04_Market_Locations                 source_url               missing=  0 / 65
07_District_Traffic_Evidence        Source URL               missing=  0 / 36
11_Old_Project_Universe_Research    population_source        missing=  0 / 19
11_Old_Project_Universe_Research    enterprise_source        missing=  4 / 19
11_Old_Project_Universe_Research    income_source            missing=  4 / 19
11_Old_Project_Universe_Research    pp_source                missing=  1 / 19
13_Additional_Context_Evidence      source_url               missing=  0 / 23


## Old-project market universe

Expected universe = 19 historical areas from verified location dataset.


In [7]:
expected_universe = {
    "Thủ Đức", "Tân Bình", "Bình Thạnh", "Quận 10", "Quận 1",
    "Quận 12", "Quận 7", "Bình Tân", "Gò Vấp", "Quận 3",
    "Tân Phú", "Dĩ An", "Quận 4", "Quận 6", "Thủ Dầu Một",
    "Vũng Tàu", "Bà Rịa", "Phú Nhuận", "Quận 5"
}

location_universe = set(locations["district_old"].dropna().astype(str))
core_universe = set(core["district_old"].dropna().astype(str))
opp_universe = set(opportunity["district_old"].dropna().astype(str))

print("Location universe:", len(location_universe))
print("Core universe:", len(core_universe))
print("Opportunity universe:", len(opp_universe))

print("\nMissing from locations:", sorted(expected_universe - location_universe))
print("Unexpected in locations:", sorted(location_universe - expected_universe))
print("Missing from core:", sorted(expected_universe - core_universe))
print("Missing from opportunity:", sorted(expected_universe - opp_universe))

assert expected_universe == location_universe, "Old-project location universe is not exactly 19 expected areas."
assert expected_universe == core_universe, "14_Complete_Screening_Core does not cover all 19 areas."
assert expected_universe == opp_universe, "09_District_Opportunity_Base does not cover all 19 areas."
assert core["district_old"].is_unique, "Duplicate district rows in screening core."
assert opportunity["district_old"].is_unique, "Duplicate district rows in opportunity base."


Location universe: 19
Core universe: 19
Opportunity universe: 19

Missing from locations: []
Unexpected in locations: []
Missing from core: []
Missing from opportunity: []


## Verified market-location audit


In [8]:
required_location_cols = {
    "branch_id","brand","branch_name","address","district_old",
    "latitude","longitude","source_url","analysis_role"
}
missing_cols = required_location_cols - set(locations.columns)
assert not missing_cols, f"Missing location columns: {missing_cols}"

print("Verified market locations:", len(locations))
print("Brands:", locations["brand"].nunique())
print("Historical areas:", locations["district_old"].nunique())
print("Missing coordinates:", int(locations[["latitude","longitude"]].isna().any(axis=1).sum()))
print("Missing source_url:", int(locations["source_url"].isna().sum()))

print("\nAnalysis roles:")
print(locations["analysis_role"].value_counts(dropna=False))

icare = locations[locations["analysis_role"].eq("ICARE_EXISTING")]
competitors = locations[locations["analysis_role"].eq("COMPETITOR_EVIDENCE")]

print("\niCare verified locations:", len(icare))
print("Competitor-evidence locations:", len(competitors))

assert len(icare) == 3, f"Expected 3 verified iCare locations, found {len(icare)}."
assert not locations[["latitude","longitude"]].isna().any(axis=None), "Missing coordinates found."


Verified market locations: 65
Brands: 10
Historical areas: 19
Missing coordinates: 0
Missing source_url: 0

Analysis roles:
analysis_role
COMPETITOR_EVIDENCE    62
ICARE_EXISTING          3
Name: count, dtype: int64

iCare verified locations: 3
Competitor-evidence locations: 62


## Coverage evidence audit


In [9]:
required_coverage_cols = {
    "branch_id","district_old","nearest_icare_km",
    "beyond_3km","beyond_5km","nearest_icare_branch"
}
missing_cols = required_coverage_cols - set(coverage.columns)
assert not missing_cols, f"Missing coverage columns: {missing_cols}"

coverage["nearest_icare_km"] = pd.to_numeric(coverage["nearest_icare_km"], errors="coerce")

print("Coverage rows:", len(coverage))
print("Missing nearest_icare_km:", int(coverage["nearest_icare_km"].isna().sum()))
print("Negative distances:", int(coverage["nearest_icare_km"].lt(0).sum()))

assert len(coverage) == len(competitors), (
    f"Coverage rows ({len(coverage)}) != competitor evidence rows ({len(competitors)})."
)
assert coverage["nearest_icare_km"].notna().all(), "Missing nearest_iCare distance."
assert not coverage["nearest_icare_km"].lt(0).any(), "Negative distance found."
assert set(coverage["beyond_3km"].dropna().astype(str).unique()).issubset({"YES","NO"})
assert set(coverage["beyond_5km"].dropna().astype(str).unique()).issubset({"YES","NO"})

display(
    coverage.groupby("district_old", as_index=False)
    .agg(
        market_points=("branch_id","count"),
        median_nearest_icare_km=("nearest_icare_km","median"),
        mean_nearest_icare_km=("nearest_icare_km","mean"),
    )
    .sort_values("median_nearest_icare_km", ascending=False)
)


Coverage rows: 62
Missing nearest_icare_km: 0
Negative distances: 0


,district_old,market_points,median_nearest_icare_km,mean_nearest_icare_km
17,Vũng Tàu,2,63.055745,63.055745
0,Bà Rịa,1,59.343154,59.343154
13,Thủ Dầu Một,2,18.519995,18.519995
8,Quận 12,4,8.767002,7.848548
2,Bình Tân,3,7.277895,7.772596
3,Dĩ An,2,5.640050,5.640050
11,Quận 6,2,5.557992,5.557992
1,Bình Thạnh,6,5.259356,5.049390
12,Quận 7,4,4.659819,4.796175
4,Gò Vấp,3,3.892399,3.908088


## Traffic evidence audit

Mục tiêu ở đây là kiểm tra **evidence coverage**, không gọi evidence count là traffic volume.


In [10]:
required_traffic_cols = {"Khu vực","Tuyến / nút","Loại evidence","Source URL","Guardrail"}
missing_cols = required_traffic_cols - set(traffic.columns)
assert not missing_cols, f"Missing traffic columns: {missing_cols}"

print("Traffic evidence rows:", len(traffic))
print("Traffic areas:", traffic["Khu vực"].nunique())
print("Distinct source URLs:", traffic["Source URL"].nunique())
print("Rows missing Source URL:", int(traffic["Source URL"].isna().sum()))

traffic_areas = set(traffic["Khu vực"].dropna().astype(str))
print("Areas with no traffic evidence:", sorted(expected_universe - traffic_areas))

display(
    traffic_summary[[
        "Khu vực",
        "Official/credible evidence rows",
        "Distinct source URLs",
        "Distinct corridors/nodes",
        "Comparable numeric traffic volume available",
        "Interpretation guardrail",
    ]]
)

assert expected_universe.issubset(set(traffic_summary["Khu vực"].dropna().astype(str))), (
    "Traffic summary does not contain all 19 old-project areas."
)


Traffic evidence rows: 36
Traffic areas: 19
Distinct source URLs: 24
Rows missing Source URL: 0
Areas with no traffic evidence: []


,Khu vực,Official/credible evidence rows,Distinct source URLs,Distinct corridors/nodes,Comparable numeric traffic volume available,Interpretation guardrail
0,Thủ Đức,1,1,1,NO,Evidence count is not traffic volume; register...
1,Tân Bình,2,2,2,NO,Evidence count is not traffic volume; register...
2,Bình Thạnh,4,4,4,NO,Evidence count is not traffic volume; register...
3,Quận 10,1,1,1,NO,Evidence count is not traffic volume; register...
4,Quận 1,1,1,1,NO,Evidence count is not traffic volume; register...
5,Quận 12,4,3,4,NO,Evidence count is not traffic volume; register...
6,Quận 7,3,2,3,NO,Evidence count is not traffic volume; register...
7,Bình Tân,4,4,4,NO,Evidence count is not traffic volume; register...
8,Gò Vấp,3,2,3,NO,Evidence count is not traffic volume; register...
9,Quận 3,1,1,1,NO,Evidence count is not traffic volume; register...


## Opportunity-base audit


In [11]:
required_opp_cols = [
    "district_old",
    "competitor_locations",
    "competitor_brands",
    "icare_locations",
    "median_nearest_icare_km",
    "uncovered_share_5km_pct",
    "official_population",
    "official_density",
    "official_acting_enterprises_2022",
    "official_enterprise_turnover_2022_billion_vnd",
    "official_avg_employee_compensation_2022_million_month",
    "traffic_evidence_rows",
    "traffic_distinct_sources",
    "traffic_distinct_corridors_nodes",
    "apartment_midpoint_proxy",
    "pp_comparability_flag",
    "population_source_quality",
    "market_data_available",
    "coverage_data_available",
    "traffic_data_available",
    "economic_data_available",
    "pp_data_available",
    "screening_readiness",
    "data_gap_note",
    "selection_guardrail",
]

missing_cols = [c for c in required_opp_cols if c not in opportunity.columns]
assert not missing_cols, f"Missing opportunity columns: {missing_cols}"

print("=== SCREENING READINESS ===")
print(opportunity["screening_readiness"].value_counts(dropna=False))

display(
    opportunity[[
        "district_old",
        "market_data_available",
        "coverage_data_available",
        "traffic_data_available",
        "economic_data_available",
        "pp_data_available",
        "screening_readiness",
        "data_gap_note",
    ]]
)


=== SCREENING READINESS ===
screening_readiness
READY_FOR_COMPARATIVE_SCREENING                      15
READY_WITH_CROSS_PROVINCE_COMPARABILITY_GUARDRAIL     2
PARTIAL                                               2
Name: count, dtype: int64


,district_old,market_data_available,coverage_data_available,traffic_data_available,economic_data_available,pp_data_available,screening_readiness,data_gap_note
0,Thủ Đức,YES,YES,YES,YES,YES,READY_FOR_COMPARATIVE_SCREENING,official traffic evidence not found yet
1,Tân Bình,YES,YES,YES,YES,YES,READY_FOR_COMPARATIVE_SCREENING,official traffic evidence not found yet
2,Bình Thạnh,YES,YES,YES,YES,YES,READY_FOR_COMPARATIVE_SCREENING,official traffic evidence not found yet
3,Quận 10,YES,YES,YES,YES,YES,READY_FOR_COMPARATIVE_SCREENING,official traffic evidence not found yet
4,Quận 1,YES,YES,YES,YES,YES,READY_FOR_COMPARATIVE_SCREENING,official traffic evidence not found yet
5,Quận 12,YES,YES,YES,YES,YES,READY_FOR_COMPARATIVE_SCREENING,official traffic evidence not found yet
6,Quận 7,YES,YES,YES,YES,YES,READY_FOR_COMPARATIVE_SCREENING,official traffic evidence not found yet
7,Bình Tân,YES,YES,YES,YES,YES,READY_FOR_COMPARATIVE_SCREENING,official traffic evidence not found yet
8,Gò Vấp,YES,YES,YES,YES,YES,READY_FOR_COMPARATIVE_SCREENING,official traffic evidence not found yet
9,Quận 3,YES,YES,YES,YES,YES,READY_FOR_COMPARATIVE_SCREENING,official traffic evidence not found yet


## Complete screening core audit

Sheet 14 dùng explicit status text thay vì fake numeric value khi nguồn không comparable.  
Vì vậy `screening_core_complete = YES` **không có nghĩa mọi metric đều là numeric và comparable**.


In [12]:
required_core_cols = [
    "district_old","competitor_locations","competitor_brands","icare_locations",
    "coverage_status","coverage_reference","population_value",
    "population_reference_year","population_quality","density_value_or_status",
    "traffic_evidence_status","traffic_evidence_rows",
    "economic_comparability_status","income_reference_status",
    "pp_proxy_status","pp_midpoint_or_status","screening_core_complete",
    "comparison_rule"
]

missing_cols = [c for c in required_core_cols if c not in core.columns]
assert not missing_cols, f"Missing core columns: {missing_cols}"

print("Core rows:", len(core))
print("Unique districts:", core["district_old"].nunique())
print("screening_core_complete:")
print(core["screening_core_complete"].value_counts(dropna=False))

blank_core = core[required_core_cols].isna().sum()
display(blank_core[blank_core > 0].to_frame("blank_cells"))

assert len(core) == 19, f"Expected 19 core rows, found {len(core)}"
assert core["screening_core_complete"].astype(str).eq("YES").all(), (
    "Some core rows are not marked complete."
)


Core rows: 19
Unique districts: 19
screening_core_complete:
screening_core_complete
YES    19
Name: count, dtype: int64


,blank_cells


## Numeric comparability audit

Không fail notebook chỉ vì external areas không có cùng HCMC enterprise/income series.  
Thay vào đó audit rõ metric nào đủ numeric và metric nào phải giữ làm guardrail.


In [13]:
numeric_candidates = [
    "competitor_locations",
    "competitor_brands",
    "icare_locations",
    "median_nearest_icare_km",
    "uncovered_share_5km_pct",
    "official_population",
    "official_density",
    "official_acting_enterprises_2022",
    "official_enterprise_turnover_2022_billion_vnd",
    "official_avg_employee_compensation_2022_million_month",
    "traffic_evidence_rows",
    "traffic_distinct_sources",
    "traffic_distinct_corridors_nodes",
    "apartment_midpoint_proxy",
]

audit = opportunity[["district_old"]].copy()

for col in numeric_candidates:
    s = pd.to_numeric(opportunity[col], errors="coerce")
    audit[col + "__available"] = s.notna()

audit["numeric_metric_count"] = audit.drop(columns=["district_old"]).sum(axis=1)

display(
    audit.sort_values(
        ["numeric_metric_count","district_old"],
        ascending=[False, True]
    )
)

print("\nIMPORTANT:")
print("- Missing remains missing; it is NOT converted to zero.")
print("- 03_EDA must use only like-for-like numeric metrics for rank/Pareto.")


,district_old,competitor_locations__available,competitor_brands__available,icare_locations__available,median_nearest_icare_km__available,uncovered_share_5km_pct__available,official_population__available,official_density__available,official_acting_enterprises_2022__available,official_enterprise_turnover_2022_billion_vnd__available,official_avg_employee_compensation_2022_million_month__available,traffic_evidence_rows__available,traffic_distinct_sources__available,traffic_distinct_corridors_nodes__available,apartment_midpoint_proxy__available,numeric_metric_count
2,Bình Thạnh,True,True,True,True,True,True,True,True,True,True,True,True,True,True,14
7,Bình Tân,True,True,True,True,True,True,True,True,True,True,True,True,True,True,14
11,Dĩ An,True,True,True,True,True,True,True,True,True,True,True,True,True,True,14
8,Gò Vấp,True,True,True,True,True,True,True,True,True,True,True,True,True,True,14
17,Phú Nhuận,True,True,True,True,True,True,True,True,True,True,True,True,True,True,14
4,Quận 1,True,True,True,True,True,True,True,True,True,True,True,True,True,True,14
3,Quận 10,True,True,True,True,True,True,True,True,True,True,True,True,True,True,14
5,Quận 12,True,True,True,True,True,True,True,True,True,True,True,True,True,True,14
9,Quận 3,True,True,True,True,True,True,True,True,True,True,True,True,True,True,14
12,Quận 4,True,True,True,True,True,True,True,True,True,True,True,True,True,True,14



IMPORTANT:
- Missing remains missing; it is NOT converted to zero.
- 03_EDA must use only like-for-like numeric metrics for rank/Pareto.


## Audit conclusion

Nếu tất cả assertions phía trên chạy qua:
- 19-area old-project universe còn nguyên;
- verified location data có đủ tọa độ;
- coverage evidence khớp competitor evidence;
- traffic summary giữ đủ 19 khu;
- opportunity/core sheets đúng schema mới;
- không có logic chọn Top 3 trong audit.

Tiếp theo chạy `02_Data_Cleaning.ipynb`.


In [14]:
# ============================================================
# CORRIDOR-LEVEL AUDIT
# ============================================================

print("=== CORRIDOR-LEVEL AUDIT ===")
print("Current phase: Top 5 district screening.")
print("Corridor/rent/site-level checks are intentionally skipped.")
print("These checks will be handled later in 04_Corridor_Comparison.ipynb.")
print("District-level audit can continue.")

=== CORRIDOR-LEVEL AUDIT ===
Current phase: Top 5 district screening.
Corridor/rent/site-level checks are intentionally skipped.
These checks will be handled later in 04_Corridor_Comparison.ipynb.
District-level audit can continue.
